- V1.1 pipeline (Schema → Text-to-SQL → Execution → Response)
- Stress Test Suite (30+ perguntas)
- Loop automatizado
- Debug completo por pergunta
- Saída estruturada para análise manual

In [1]:
# =============================================================================
# DATA AGENT V1.1 + STRESS TEST SUITE
# =============================================================================
#
# ARQUITETURA
#
# Question
#    ↓
# Schema
#    ↓
# Text-to-SQL
#    ↓
# Execution
#    ↓
# Natural Language Response
#
# + STRESS TEST AUTOMATION
#
# Objetivo:
# Avaliar robustez do Text-to-SQL em múltiplos cenários
#
# =============================================================================


# =============================================================================
# STEP 1 - IMPORTS
# =============================================================================

import sqlite3
import pandas as pd
from langchain_ollama import ChatOllama


# =============================================================================
# STEP 2 - DATABASE
# =============================================================================

conn = sqlite3.connect("oil.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS well_production")

cursor.execute("""
CREATE TABLE well_production (
    well_name TEXT,
    field_name TEXT,
    production_date TEXT,
    oil_bbl REAL,
    gas_mscf REAL,
    water_bbl REAL,
    hours_on REAL
)
""")

rows = [
    ("WELL-A1","FIELD-X","2026-06-01",1200,800,300,24),
    ("WELL-A2","FIELD-X","2026-06-01",900,600,500,24),
    ("WELL-B1","FIELD-Y","2026-06-01",1500,1100,200,24),

    ("WELL-A1","FIELD-X","2026-06-02",1250,820,320,24),
    ("WELL-A2","FIELD-X","2026-06-02",920,620,510,24),
    ("WELL-B1","FIELD-Y","2026-06-02",1520,1120,210,24),

    ("WELL-A1","FIELD-X","2026-06-03",1230,810,310,24),
    ("WELL-A2","FIELD-X","2026-06-03",910,610,505,24),
    ("WELL-B1","FIELD-Y","2026-06-03",1550,1150,220,24)
]

cursor.executemany("""
INSERT INTO well_production VALUES (?,?,?,?,?,?,?)
""", rows)

conn.commit()

print("Banco criado com sucesso")


# =============================================================================
# STEP 3 - SCHEMA
# =============================================================================

schema_df = pd.read_sql("PRAGMA table_info(well_production)", conn)
schema_text = "\n".join(schema_df["name"].tolist())


print("\nSCHEMA:")
print(schema_text)


# =============================================================================
# STEP 4 - LLM
# =============================================================================

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)


# =============================================================================
# STEP 5 - STRESS TEST SUITE
# =============================================================================

test_questions = [

    "Qual poço teve maior produção acumulada de óleo?",
    "Qual poço produziu mais água?",
    "Qual poço produziu mais gás?",
    "Qual campo produziu mais óleo?",
    "Qual campo produziu mais gás?",
    "Qual campo produziu mais água?",
    "Qual poço teve menor produção de óleo?",
    "Qual poço teve maior produção média de óleo?",
    "Qual poço teve maior produção média de gás?",
    "Qual poço teve maior produção média de água?",
    "Mostre os três poços com maior produção de óleo.",
    "Mostre os três poços com maior produção de gás.",
    "Mostre os três poços com maior produção de água.",
    "Qual foi a produção total de óleo do FIELD-X?",
    "Qual foi a produção total de óleo do FIELD-Y?",
    "Qual poço apresentou maior GOR?",
    "Qual poço apresentou menor GOR?",
    "Qual poço apresentou maior Water Cut?",
    "Qual poço apresentou menor Water Cut?",
    "Qual foi o total de óleo produzido no dia 2026-06-02?",
    "Qual foi o total de gás produzido no dia 2026-06-02?",
    "Qual foi o total de água produzida no dia 2026-06-02?",
    "Qual poço teve mais horas de operação?",
    "Qual poço teve menos horas de operação?",
    "Qual campo possui mais poços?",
    "Qual foi a maior produção diária de óleo?",
    "Qual foi a maior produção diária de gás?",
    "Qual foi a maior produção diária de água?",
    "Liste todas as datas disponíveis.",
    "Quantos registros existem na tabela?"
]


# =============================================================================
# STEP 6 - PIPELINE FUNCTION
# =============================================================================

def run_pipeline(question: str):

    # -----------------------------
    # TEXT TO SQL PROMPT
    # -----------------------------
    sql_prompt = f"""
Você é especialista em SQL.

Tabela:
well_production

Colunas:
{schema_text}

REGRAS:
- Retorne SOMENTE SQL
- Não use explicações
- Não use markdown

Pergunta:
{question}
"""

    response = llm.invoke(sql_prompt)
    sql = response.content.strip()

    # -----------------------------
    # EXECUTION
    # -----------------------------
    try:
        result_df = pd.read_sql(sql, conn)
    except Exception as e:
        result_df = None
        print("ERRO SQL:", e)

    # -----------------------------
    # RESPONSE LAYER
    # -----------------------------
    if result_df is not None and not result_df.empty:

        result_text = result_df.to_string(index=False)

        response_prompt = f"""
Você é um analista de produção de petróleo.

Pergunta:
{question}

Resultado SQL:
{result_text}

Gere uma resposta curta e clara.
Não invente dados.
"""

        final_response = llm.invoke(response_prompt).content

    else:
        final_response = "Sem resultado ou erro na execução."

    return sql, result_df, final_response


# =============================================================================
# STEP 7 - STRESS TEST EXECUTION LOOP
# =============================================================================

print("\n")
print("=" * 80)
print("STRESS TEST START")
print("=" * 80)

results_log = []

for i, question in enumerate(test_questions):

    print("\n" + "=" * 80)
    print(f"QUESTION {i+1}")
    print("=" * 80)
    print(question)

    sql, df, response = run_pipeline(question)

    print("\nSQL GERADO:")
    print(sql)

    print("\nRESULTADO:")
    print(df)

    print("\nRESPOSTA:")
    print(response)

    results_log.append({
        "question": question,
        "sql": sql,
        "result": str(df),
        "response": response
    })


# =============================================================================
# STEP 8 - SUMMARY
# =============================================================================

print("\n")
print("=" * 80)
print("STRESS TEST FINISHED")
print("=" * 80)

print(f"Total de testes: {len(test_questions)}")


# =============================================================================
# STEP 9 - CLEANUP
# =============================================================================

conn.close()

print("\nConexão encerrada")

Banco criado com sucesso

SCHEMA:
well_name
field_name
production_date
oil_bbl
gas_mscf
water_bbl
hours_on


STRESS TEST START

QUESTION 1
Qual poço teve maior produção acumulada de óleo?

SQL GERADO:
SELECT well_name FROM ( SELECT well_name, SUM(oil_bbl) AS total_oil FROM well_production GROUP BY well_name ) ORDER BY total_oil DESC LIMIT 1

RESULTADO:
  well_name
0   WELL-B1

RESPOSTA:
O poço com a maior produção acumulada de óleo é o WELL-B1.

QUESTION 2
Qual poço produziu mais água?

SQL GERADO:
SELECT well_name FROM well_production ORDER BY water_bbl DESC LIMIT 1

RESULTADO:
  well_name
0   WELL-A2

RESPOSTA:
O poço que produziu mais água foi o WELL-A2.

QUESTION 3
Qual poço produziu mais gás?

SQL GERADO:
SELECT well_name FROM well_production ORDER BY gas_mscf DESC LIMIT 1

RESULTADO:
  well_name
0   WELL-B1

RESPOSTA:
O poço que produziu mais gás foi o WELL-B1.

QUESTION 4
Qual campo produziu mais óleo?

SQL GERADO:
SELECT well_name FROM well_production GROUP BY well_name ORDER B